# WRAP_P6_PATTERN_MINING — Notebook 04

**Spec ref:** §9.2 — Unsupervised L3 pattern discovery via HDBSCAN.

Promotion is a *human* decision in the UI (§10.3); this notebook only writes candidates.

## Section 00 — Env & Imports

In [1]:
import sys, logging
from pathlib import Path
import numpy as np
import pandas as pd

# ── shared config ────────────────────────────────────────────────────────────
_NB_DIR = Path('<repo>/p6lab/notebooks')
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))
import _common
from _common import (
    NOTEBOOK_DATA_SLICE, MIN_CLUSTER_SIZE, MIN_SAMPLES,
    collect_events, versioned_dir,
)
_DS       = NOTEBOOK_DATA_SLICE
SYMBOL    = _DS['symbol']
TICK_SIZE = _DS['tick_size']

_P6LAB        = Path(_common._P6LAB_ROOT)
ARTIFACTS_DIR = _P6LAB / 'artifacts' / 'p6lab'

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

from p6lab.ingestion.event_windowing import (
    BurstDetectorConfig, WindowAnchorStrategy, WindowIterator,
)
from p6lab.patterns.miner import (
    MinedCandidate, SHAPE_VECTOR_DIM, compute_umap_projection,
    extract_event_shape_vector, filter_candidates, run_hdbscan,
)
from p6lab.patterns.library import PatternLibrary

np.random.seed(42)
print(f'Data: {_DS["data_file"]}  symbol={SYMBOL}')


Data: <repo>/data/nq-mbo-sample-15min.dbn.zst  symbol=NQ


## Section 01 — Load Triple-View Parquet

In [2]:
mbo_orders = await collect_events(_DS)
# Convert Order → dict shape expected by miner (timestamp_ms, side, action, price, size)
events = [
    {
        'timestamp_ms': o.timestamp_ms,
        'side': o.side.value if hasattr(o.side, 'value') else str(o.side),
        'action': o.action.value if hasattr(o.action, 'value') else str(o.action),
        'price': o.price,
        'size': o.size,
        'order_id': o.order_id,
    }
    for o in mbo_orders
]
log.info('01 │ %d L3 events from MBO stream', len(events))
assert len(events) > 100, '01 │ too few MBO events for pattern mining'


Opening MBO data from <repo>/data/nq-mbo-sample-15min.dbn.zst (streaming mode)


Streaming ready. instrument_id=42004058


01 │ 32240 L3 events from MBO stream


## Section 02 — Burst Windowing

In [3]:
# Default from sanity/burst_tuning_sweep.py — see artifacts/p6lab/mining/burst_tuning_report.md
cfg = BurstDetectorConfig(min_events_per_100ms=50, lookback_ms=500,
                          lookahead_ms=1500, min_burst_gap_ms=200)
windows = list(WindowIterator.create(WindowAnchorStrategy.BURST_ANCHORED,
                                     events=events, burst_config=cfg))
log.info('02 │ %d burst windows detected', len(windows))
assert len(windows) > 0, '02 │ no bursts found — relax BurstDetectorConfig thresholds'


02 │ 129 burst windows detected


## Section 03 — 30-dim Event Shape Extraction

In [4]:
if windows:
    vectors = np.stack([extract_event_shape_vector(w.events) for w in windows])
else:
    vectors = np.zeros((0, SHAPE_VECTOR_DIM))
log.info('03 │ shape matrix %s', vectors.shape)


03 │ shape matrix (129, 30)


## Section 04 — UMAP Projection (visualization only)

In [5]:
if len(vectors) >= 4:
    proj = compute_umap_projection(vectors, n_components=2)
    log.info('04 │ UMAP %s', proj.shape)
else:
    log.info('04 │ skipped — need ≥4 windows')


04 │ UMAP (129, 2)


## Section 05 — HDBSCAN Clustering

In [6]:
labels, _ = run_hdbscan(vectors, min_cluster_size=5, min_samples=3)
n_clusters = len(set(int(l) for l in labels if l >= 0))
log.info('05 │ %d clusters', n_clusters)


05 │ 2 clusters


## Section 06 — Cluster Profiling (skeleton)

Per-cluster: member count, exemplars, centroid, intra-cluster variance.

In [7]:
# Per-cluster profiling: size, centroid, intra-cluster dispersion
if n_clusters > 0:
    rows = []
    centroids = {}
    for cl in sorted(set(int(l) for l in labels if l >= 0)):
        mask = labels == cl
        members = vectors[mask]
        centroid = members.mean(axis=0)
        centroids[cl] = centroid
        # Intra-cluster RMS distance to centroid
        rms = float(np.sqrt(((members - centroid) ** 2).sum(axis=1).mean()))
        rows.append({
            'cluster': cl, 'n_members': int(mask.sum()),
            'rms_intra_dist': rms,
            'centroid_norm': float(np.linalg.norm(centroid)),
        })
    cluster_stats = pd.DataFrame(rows).sort_values('n_members', ascending=False)
    log.info('06 │ cluster stats:\n' + cluster_stats.to_string(index=False))
else:
    cluster_stats = pd.DataFrame()
    centroids = {}
    log.info('06 │ no clusters — HDBSCAN labeled all points as noise')
cluster_stats


06 │ cluster stats:
 cluster  n_members  rms_intra_dist  centroid_norm
       0        102       ▒.▒▒     ▒.▒▒
       1          6       ▒.▒▒     ▒.▒▒


,cluster,n_members,rms_intra_dist,centroid_norm
0,0,102,▒.▒▒,▒.▒▒
1,1,6,▒.▒▒,▒.▒▒


## Section 07 — Forward-Outcome Labeling (skeleton)

Use `labeler.label_pattern_instance` per cluster member to compute 1m/5m/15m/1h forward outcomes.

In [8]:
# Forward-outcome surface: per cluster, measure mean Δprice over next 60s
HORIZON_MS = 60_000
forward_outcomes = []
ts_arr = np.array([e['timestamp_ms'] for e in events])
px_arr = np.array([e['price'] for e in events])
if n_clusters > 0:
    for cl in sorted(centroids):
        # Anchor timestamps for each cluster member
        member_anchors = [windows[i].anchor_ms for i in np.where(labels == cl)[0]]
        moves = []
        for anchor in member_anchors:
            # Find first event >= anchor and first event >= anchor+HORIZON
            i0 = int(np.searchsorted(ts_arr, anchor))
            i1 = int(np.searchsorted(ts_arr, anchor + HORIZON_MS))
            if i0 < len(px_arr) and i1 > i0 and i1 <= len(px_arr):
                moves.append(px_arr[i1 - 1] - px_arr[i0])
        if moves:
            arr = np.array(moves)
            forward_outcomes.append({
                'cluster': cl, 'n': len(arr),
                'mean_move_ticks': float(arr.mean() / TICK_SIZE),
                'hit_rate_up': float((arr > 0).mean()),
                'sharpe_proxy': float(arr.mean() / (arr.std(ddof=0) + 1e-9)),
            })
    forward_df = pd.DataFrame(forward_outcomes)
    log.info('07 │ forward outcomes (60s):\n' + forward_df.to_string(index=False))
else:
    forward_df = pd.DataFrame()
    log.info('07 │ skipped — no clusters')
forward_df


07 │ forward outcomes (60s):
 cluster   n  mean_move_ticks  hit_rate_up  sharpe_proxy
       0 102       ▒.▒▒     ▒.▒▒     ▒.▒▒
       1   6       ▒.▒▒     ▒.▒▒      ▒.▒▒


,cluster,n,mean_move_ticks,hit_rate_up,sharpe_proxy
0,0,102,▒.▒▒,▒.▒▒,▒.▒▒
1,1,6,▒.▒▒,▒.▒▒,▒.▒▒


## Section 08 — Distance-to-Existing Patterns (skeleton)

Cosine distance from each candidate centroid to active library centroids.

In [9]:
# Distance-to-existing: compare each new centroid to active library patterns
from numpy.linalg import norm
lib_path = ARTIFACTS_DIR / 'pattern_library' / 'library.yaml'
active = {}
if lib_path.exists():
    lib = PatternLibrary(lib_path); lib.load()
    active = lib.get_active_patterns()  # dict[str, PatternDefinition]
log.info('08 │ %d active library patterns', len(active))

rows = []
if active and centroids:
    for cl, c in centroids.items():
        for pat_id, pat in active.items():
            ref = np.array(getattr(pat, 'centroid', None) or [])
            if ref.size == 0 or ref.shape != c.shape:
                continue
            cos_sim = float(np.dot(c, ref) / (norm(c) * norm(ref) + 1e-9))
            rows.append({
                'new_cluster': cl, 'library_pattern': pat_id,
                'cosine_sim': cos_sim, 'cosine_dist': 1.0 - cos_sim,
            })
if rows:
    dist_df = pd.DataFrame(rows).sort_values(['new_cluster', 'cosine_dist'])
    log.info('08 │ distance to library:\n' + dist_df.to_string(index=False))
else:
    dist_df = pd.DataFrame()
    log.info('08 │ no library patterns with comparable centroid dimensions')
dist_df


08 │ 2 active library patterns


08 │ no library patterns with comparable centroid dimensions


""


## Section 09 — Candidate Filter Pipeline

In [ ]:
candidates = []
if not forward_df.empty:
    for _, r in forward_df.iterrows():
        candidates.append(MinedCandidate(
            cluster_id=int(r['cluster']),
            centroid=centroids[int(r['cluster'])],
            member_count=int(r['n']),
            exemplar_timestamps_ms=[],
            outcome_stats={},
            hit_rate_5m=float(r['hit_rate_up']),
            sharpe=float(r['sharpe_proxy']),
            cosine_distance_to_nearest_existing=1.0,
            random_state=42,
        ))
existing_arr = []
if lib_path.exists():
    lib = PatternLibrary(lib_path); lib.load()
    for pat in lib.get_active_patterns().values():
        c = getattr(pat, 'centroid', None)
        if c: existing_arr.append(np.array(c))

# Adaptive min_occurrences: production uses MIN_OCCURRENCES=200 (hours of
# tape, clusters need >=200 members for statistical power). At smoketest
# scale we'd silently empty the pipeline — use max(10, n_windows/20) so at
# least one candidate surfaces. Audit item B.
n_windows = len(windows) if windows else 0
min_occ = max(10, n_windows // 20)
log.info('09 │ filter min_occurrences=%d (n_windows=%d)', min_occ, n_windows)

kept = filter_candidates(
    candidates, existing_centroids=existing_arr or None,
    min_occurrences_override=min_occ,
)
log.info('09 │ %d candidates after filter (from %d clusters)',
         len(kept), len(candidates))
kept_df = pd.DataFrame([{'cluster_id': k.cluster_id, 'member_count': k.member_count, 'hit_rate_5m': k.hit_rate_5m, 'sharpe': k.sharpe} for k in kept])
kept_df


## Section 10 — Write Candidates (full mine() orchestrator)

In [11]:
out_dir = versioned_dir(ARTIFACTS_DIR / 'mining', 'nb04',
                        data_slice=_DS,
                        extra={'n_clusters': int(n_clusters),
                               'n_candidates': len(candidates),
                               'n_kept': len(kept)})
if not cluster_stats.empty:
    cluster_stats.to_csv(out_dir / 'cluster_stats.csv', index=False)
if not forward_df.empty:
    forward_df.to_csv(out_dir / 'forward_outcomes.csv', index=False)
if not kept_df.empty:
    kept_df.to_parquet(out_dir / 'mined_candidates.parquet', index=False)
log.info('10 │ wrote artefacts to %s', out_dir)


10 │ wrote artefacts to <repo>/p6lab/artifacts/p6lab/mining/nb04_20260420_1648
